# Week 3 — V1 Data Pipeline · FYPilot (AI FYP Topic Recommendation Assistant)

**Pathway:** A — PDF/TXT only (single modality: documents). Pathways B–F are not used.

**Data:** synthetic practice dataset of 3 documents modelled on FYPilot's real sources (lab practice-dataset option).
Practice results do **not** show how well FYPilot itself will perform.

| Source | File | Supports requirement | Real source that will replace it |
| --- | --- | --- | --- |
| S01 | `fai_fyp_guidelines_synthetic.pdf` | FR5 (answer FYP rules / timeline), RAI3 (integrity rules) | FAI FYP Guidelines (UTM FAI FYP site) |
| S02 | `past_fyp_abstracts_synthetic.txt` | FR3 (show similar past FYPs / flag duplicates) | Past thesis abstracts, UTM ePrints |
| S03 | `staff_research_profiles_synthetic.txt` | FR2, NFR3 (match lecturers by research area) | UTM staff profile / FAI research group pages |

RAI1 (cite a verifiable source) applies to all three, so every prepared record keeps its source file and SHA-256 checksum.

**Run order:** Cell 1 → upload the 3 files → Cell 3 → A3 inspection → Cell 4 → Cell 4A → Cell 5A1 → B3 comparison → Cell 5A2 → B3 output check → B4.1 → Cell 6A → Cell 6B → Cell 7 → Cell 8A → Cell 9 → C4a export.

## Part A — Start with the Data
### Step A2 — Workspace (Cell 1, REQUIRED)

In [1]:
# Cell 1 — Workspace setup (RUN FIRST)
from pathlib import Path

BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Compatibility aliases for notebook examples
RAW = RAW_DIR
PROCESSED = PROCESSED_DIR
BASE = BASE_DIR
OUT = PROCESSED_DIR

print("Raw folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)

Raw folder: /content/week3_project/data/raw
Processed folder: /content/week3_project/data/processed


**Upload:** in the Colab Files panel open `week3_project/data/raw` → ⋮ → Upload, and upload the three files from the repo's `data/raw/` folder.
Cell 2 (dummy ZIP extraction) is **not used** — this project uploads its own practice files.

In [2]:
# Cell 3 — Verify uploaded inputs (self-contained)
from pathlib import Path
RAW = Path("/content/week3_project/data/raw")
RAW_DIR = RAW
BASE_DIR = RAW.parent.parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
BASE = BASE_DIR
OUT = PROCESSED_DIR

files = sorted(f for f in RAW.rglob("*") if f.is_file())
print("Input count:", len(files))
for f in files:
    print(f.relative_to(RAW), "bytes:", f.stat().st_size)

assert files, "Upload at least one relevant input file before proceeding."
print("Upload verified. Select your relevant input(s) for Step A3.")

Input count: 3
fai_fyp_guidelines_synthetic.pdf bytes: 4568
past_fyp_abstracts_synthetic.txt bytes: 5059
staff_research_profiles_synthetic.txt bytes: 1872
Upload verified. Select your relevant input(s) for Step A3.


### Step A3 — Inspect the selected sources (REQUIRED)
Inspect all three sources and at least two representative records, not just a "file loaded" message.

In [3]:
# A3 — Inspection evidence
!pip -q install pypdf
import re
from pypdf import PdfReader

SOURCES = {
    "guidelines":     "fai_fyp_guidelines_synthetic.pdf",       # S01 — FR5, RAI3
    "past_fyps":      "past_fyp_abstracts_synthetic.txt",       # S02 — FR3
    "staff_profiles": "staff_research_profiles_synthetic.txt",  # S03 — FR2, NFR3
}

def read_source(path):
    if path.suffix.lower() == ".pdf":
        reader = PdfReader(str(path))
        pages = [(i + 1, p.extract_text() or "") for i, p in enumerate(reader.pages)]
    elif path.suffix.lower() == ".txt":
        pages = [(1, path.read_text(encoding="utf-8"))]
    else:
        raise ValueError("Pathway A needs a PDF or TXT source")
    return pages

for key, rel in SOURCES.items():
    path = RAW_DIR / rel
    pages = read_source(path)
    text = "\n".join(t for _, t in pages)
    blank_line_paragraphs = len([p for p in re.split(r"\n\s*\n", text) if p.strip()])
    print(f"=== {key}: {rel}")
    print("Type:", path.suffix, "| Pages/segments:", len(pages), "| Characters:", len(text),
          "| Blank-line paragraphs:", blank_line_paragraphs)
    print("Sample:", repr(text[:220]))
    print()

# Representative records from the semi-structured TXT sources
past = (RAW_DIR / SOURCES["past_fyps"]).read_text(encoding="utf-8")
staff = (RAW_DIR / SOURCES["staff_profiles"]).read_text(encoding="utf-8")
past_records = [b for b in re.split(r"\n\s*\n", past) if b.startswith("Record ID")]
staff_records = [b for b in re.split(r"\n\s*\n", staff) if b.startswith("Staff ID")]
print("Past-FYP records:", len(past_records), "| fields:", [l.split(':')[0] for l in past_records[0].splitlines()])
print("Staff records:", len(staff_records), "| fields:", [l.split(':')[0] for l in staff_records[0].splitlines()])
print("\nRecord 1 (past FYP):\n" + past_records[0][:400])
print("\nRecord 2 (staff profile):\n" + staff_records[0])

=== guidelines: fai_fyp_guidelines_synthetic.pdf
Type: .pdf | Pages/segments: 2 | Characters: 3313 | Blank-line paragraphs: 2
Sample: 'Faculty of Artificial Intelligence - Final Year Project\n (FYP) Guidelines (Synthetic)\nSYNTHETIC PRACTICE DATA - created for the AI Engineering Topic 3 lab. Not official UTM\ncontent.\nVersion 1.0 (synthetic), Semester 1 20'



=== past_fyps: past_fyp_abstracts_synthetic.txt
Type: .txt | Pages/segments: 1 | Characters: 5059 | Blank-line paragraphs: 9
Sample: 'SYNTHETIC PRACTICE DATA - created for the AI Engineering Topic 3 lab. Not official UTM content.\nSource: synthetic records modelled on UTM ePrints abstract pages (title, year, area, abstract).\n\nRecord ID: PFYP-2023-01\nTit'

=== staff_profiles: staff_research_profiles_synthetic.txt
Type: .txt | Pages/segments: 1 | Characters: 1872 | Blank-line paragraphs: 6
Sample: 'SYNTHETIC PRACTICE DATA - created for the AI Engineering Topic 3 lab. Not official UTM content.\nSource: synthetic records modelled on UTM staff profile and FAI research group pages. Lecturer names are placeholders.\n\nStaf'

Past-FYP records: 8 | fields: ['Record ID', 'Title', 'Year', 'Research area', 'Abstract']
Staff records: 5 | fields: ['Staff ID', 'Name', 'Department', 'Research areas', 'Supervision']

Record 1 (past FYP):
Record ID: PFYP-2023-01
Title: Deep Learning-Based Durian Leaf

## Part B — Prepare the Data
### Step B2 — Common helpers (Cell 4 and Cell 4A, REQUIRED)

In [4]:
# Cell 4 — REQUIRED: Source Selection + Lineage Utilities
# Run ONCE after Cell 1 and Cell 3

from pathlib import Path
import json
import hashlib
from datetime import datetime, timezone

# Select the exact source before parsing.
# Never automatically take the first file by extension.

def select_source(relative_path=None, filename=None, suffixes=None):
    if relative_path:
        selected = RAW_DIR / relative_path
        if not selected.is_file():
            raise FileNotFoundError(f"Source not found: {selected}")
        return selected
    matches = sorted(
        p for p in RAW_DIR.rglob("*")
        if p.is_file()
        and (filename is None or p.name == filename)
        and (suffixes is None or p.suffix.lower() in suffixes)
    )
    print("Candidates:", [str(p.relative_to(RAW_DIR)) for p in matches])
    if len(matches) != 1:
        raise ValueError(
            "Select exactly one source. "
            "Set relative_path to an exact file under RAW_DIR."
        )
    return matches[0]

def source_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def save_manifest(records):
    manifest = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "records": records
    }
    target = BASE_DIR / "data" / "week3_manifest.json"
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
    print("Saved lineage:", target)
    return manifest

# Confirmation messages
print("Cell 4 completed successfully.")
print("Source selection and lineage utilities are ready.")
print("Available functions:")
print("- select_source()")
print("- source_hash()")
print("- save_manifest()")

Cell 4 completed successfully.
Source selection and lineage utilities are ready.
Available functions:
- select_source()
- source_hash()
- save_manifest()


In [5]:
# CELL 4A — COMMON LINEAGE REGISTRY
# REQUIRED: Run once before selected Part B pathways
from pathlib import Path
lineage_entries = {}

def register_lineage(source, output, processing):
    source = Path(source).resolve()
    output = Path(output).resolve()
    # Validate source
    if not source.is_file():
        raise FileNotFoundError(f"Source file not found: {source}")
    # Validate prepared output
    if not output.is_file():
        raise FileNotFoundError(f"Prepared output not found: {output}")
    if output.stat().st_size == 0:
        raise ValueError(f"Prepared output is empty: {output}")
    # Validate processing description
    if not isinstance(processing, str) or not processing.strip():
        raise ValueError("A processing description is required.")
    processing = processing.strip()
    if processing.upper().startswith("REPLACE"):
        raise ValueError("Replace the placeholder with the actual method.")
    # Ensure source and output belong to project workspace
    source.relative_to(RAW_DIR.resolve())
    output.relative_to(PROCESSED_DIR.resolve())
    # Same source/output pair is updated, not duplicated
    key = (str(source), str(output))
    lineage_entries[key] = {"source": source, "output": output, "processing": processing}
    print("Registered:", source.name, "->", output.name)

print("Common lineage registry ready.")
print("Registered entries:", len(lineage_entries))

Common lineage registry ready.
Registered entries: 0


### Pathway A1 — Extract, clean and chunk (CONDITIONAL REQUIRED: PDF/TXT)
Same extraction, whitespace cleaning, fixed and paragraph-aware chunkers as the lab code, applied to each selected source.
**One justified addition:** `restore_heading_breaks()` — A3 showed the PDF extracts with no blank lines between sections,
so the paragraph-aware splitter could not see any boundaries. It inserts a blank line before numbered section headings (`1. `, `2. ` …). No text is removed or reworded.

In [6]:
# Cell 5A1 — PATHWAY A1 — PDF/TXT
# CONDITIONAL REQUIRED: Run only for document-based projects
!pip -q install pypdf

from pypdf import PdfReader
import re

# Step 1: Select the exact source files (explicit paths, no auto-pick)
selected = {key: select_source(relative_path=rel) for key, rel in SOURCES.items()}

# Step 2: Extract text according to file type
def extract_text(f):
    if f.suffix.lower() == ".pdf":
        reader = PdfReader(str(f))
        pages = [(i + 1, page.extract_text() or "") for i, page in enumerate(reader.pages)]
    elif f.suffix.lower() == ".txt":
        pages = [(1, f.read_text(encoding="utf-8"))]
    else:
        raise ValueError("Pathway A needs a PDF or TXT source")
    return pages

# Step 3: Basic text cleaning (lab) + restore section breaks (justified addition)
def basic_clean(text):
    return re.sub(r"[ \t]+", " ", text).strip()

def restore_heading_breaks(text):
    # Blank line before numbered section headings such as "4. Timeline for FYP 1"
    return re.sub(r"\n(?=\d+\. [A-Z])", "\n\n", text)

# Step 4: Fixed-size chunking
def fixed_chunks(s, size=500):
    if size <= 0:
        raise ValueError("Chunk size must be positive")
    return [s[i:i + size] for i in range(0, len(s), size)]

# Step 5: Paragraph-aware chunking
def paragraph_chunks(s, size=500):
    if size <= 0:
        raise ValueError("Chunk size must be positive")
    parts = []
    current = ""
    for segment in re.split(r"\n\s*\n", s):
        segment = segment.strip()
        if not segment:
            continue
        if len(segment) > size:
            if current:
                parts.append(current)
                current = ""
            parts.extend(fixed_chunks(segment, size))
        elif len(current) + len(segment) + (2 if current else 0) <= size:
            current += ("\n\n" if current else "") + segment
        else:
            if current:
                parts.append(current)
            current = segment
    if current:
        parts.append(current)
    return parts

clean_text = {}
for key, f in selected.items():
    pages = extract_text(f)
    text = "\n".join(content for _, content in pages)
    clean = basic_clean(text)
    print("Source:", f.name, "| Pages/segments:", len(pages),
          "| Extracted characters:", len(text), "| Cleaned characters:", len(clean))
    assert clean, "No usable text: check extraction or OCR needs."
    clean_text[key] = clean

print("\nPATHWAY A1 completed successfully.")

Source: fai_fyp_guidelines_synthetic.pdf | Pages/segments: 2 | Extracted characters: 3313 | Cleaned characters: 3312
Source: past_fyp_abstracts_synthetic.txt | Pages/segments: 1 | Extracted characters: 5059 | Cleaned characters: 5058
Source: staff_research_profiles_synthetic.txt | Pages/segments: 1 | Extracted characters: 1872 | Cleaned characters: 1871

PATHWAY A1 completed successfully.


### Step B3 — Compare preparation configurations (same input data)
Configuration A is the lab default (fixed 500 characters). B–D are paragraph-aware variants.
Evidence: chunk count, chunks that end mid-sentence, and FYP / staff records split across chunks
(a split record cannot be cited as one source for FR3 / FR2 / RAI1).

In [7]:
# B3 — Comparison evidence
import pandas as pd

CONFIGS = {
    "A: fixed-500 (lab default)":            lambda t: fixed_chunks(t, 500),
    "B: paragraph-aware-500":                lambda t: paragraph_chunks(t, 500),
    "C: paragraph-aware-800 + heading breaks":  lambda t: paragraph_chunks(restore_heading_breaks(t), 800),
    "D: paragraph-aware-1000 + heading breaks": lambda t: paragraph_chunks(restore_heading_breaks(t), 1000),
}

def ends_mid_sentence(chunk):
    return not re.search(r"[.!?:)]\s*$", chunk.strip())

def records_split(text, chunks):
    blocks = [b.strip() for b in re.split(r"\n\s*\n", text) if re.match(r"(Record ID|Staff ID):", b.strip())]
    return sum(1 for b in blocks if not any(b in c for c in chunks)), len(blocks)

rows = []
for key, text in clean_text.items():
    for name, fn in CONFIGS.items():
        chunks = fn(text)
        split, total = records_split(text, chunks)
        rows.append({
            "source": key, "configuration": name, "chunks": len(chunks),
            "mid_sentence_ends": sum(ends_mid_sentence(c) for c in chunks),
            "records_split": f"{split}/{total}" if total else "n/a",
            "max_chars": max(len(c) for c in chunks),
        })
comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False))

print("\n--- Boundary example: past_fyps, first FYP record ---")
print("A fixed-500   :", repr(CONFIGS["A: fixed-500 (lab default)"](clean_text["past_fyps"])[1][-120:]))
print("C para-800+hb :", repr(CONFIGS["C: paragraph-aware-800 + heading breaks"](clean_text["past_fyps"])[1][-120:]))

        source                            configuration  chunks  mid_sentence_ends records_split  max_chars
    guidelines               A: fixed-500 (lab default)       7                  6           n/a        500
    guidelines                   B: paragraph-aware-500       8                  6           n/a        500
    guidelines  C: paragraph-aware-800 + heading breaks       7                  0           n/a        596
    guidelines D: paragraph-aware-1000 + heading breaks       5                  0           n/a        876
     past_fyps               A: fixed-500 (lab default)      11                  6           8/8        500
     past_fyps                   B: paragraph-aware-500      17                  8           8/8        500
     past_fyps  C: paragraph-aware-800 + heading breaks       9                  0           0/8        708
     past_fyps D: paragraph-aware-1000 + heading breaks       8                  0           0/8        901
staff_profiles              

**Decision:** configuration **C — paragraph-aware, 800 characters, with heading breaks restored.**
It is the only setting with zero mid-sentence cuts *and* zero split records while keeping one record or section per chunk (D merges several lecturers into one chunk, which weakens per-source citation).
Accepted limitation: chunks are larger and uneven, and page numbers are not preserved (`page = None`).

In [8]:
# Cell 5A2 — PATHWAY A2 — REQUIRED after A1
# Choose and export traceable text chunks
import json

# Step 1: Select the preparation approach (from B3 evidence)
CHOSEN_METHOD = "paragraph-aware"
CHUNK_SIZE = 800
RESTORE_HEADING_BREAKS = True
approach = f"paragraph-aware-char-{CHUNK_SIZE}; restore_heading_breaks={RESTORE_HEADING_BREAKS}; whitespace-normalised"

# Step 2: Create traceable records for each selected source
records = []
for key, f in selected.items():
    text = restore_heading_breaks(clean_text[key]) if RESTORE_HEADING_BREAKS else clean_text[key]
    chunks = paragraph_chunks(text, CHUNK_SIZE) if CHOSEN_METHOD == "paragraph-aware" else fixed_chunks(text, CHUNK_SIZE)
    for i, chunk in enumerate(chunks, 1):
        records.append({
            "record_id": f"{f.stem}-{i:04d}",
            "text": chunk,
            "source": str(f.relative_to(RAW_DIR)),
            "source_type": key,
            "source_sha256": source_hash(f),
            "processing": approach,
            "page": None
        })
# Page is None because the basic whole-document splitter does not preserve page boundaries.

# Step 3: Export prepared records
prepared_path = PROCESSED_DIR / "prepared_text.jsonl"
prepared_path.parent.mkdir(parents=True, exist_ok=True)
with prepared_path.open("w", encoding="utf-8") as output:
    for row in records:
        output.write(json.dumps(row, ensure_ascii=False) + "\n")

# Step 4: Basic validation
assert prepared_path.is_file(), "Prepared output not found"
assert prepared_path.stat().st_size > 0, "Prepared output is empty"
assert records, "No text chunks were produced"
assert all(row["text"].strip() for row in records), "One or more chunks are empty"
with prepared_path.open("r", encoding="utf-8") as file:
    exported_records = [json.loads(line) for line in file if line.strip()]
assert len(exported_records) == len(records), "Exported record count does not match"

# Step 5: Display evidence
print("PATHWAY A2 completed successfully.")
print("Selected approach:", approach)
print("Total chunks exported:", len(records))
print("Chunks per source:", pd.Series([r["source_type"] for r in records]).value_counts().to_dict())
print("Prepared file:", prepared_path)
print("File size:", prepared_path.stat().st_size, "bytes")
print("Validation: PASS")
print("First record:", records[0])

PATHWAY A2 completed successfully.
Selected approach: paragraph-aware-char-800; restore_heading_breaks=True; whitespace-normalised
Total chunks exported: 19
Chunks per source: {'past_fyps': 9, 'guidelines': 7, 'staff_profiles': 3}
Prepared file: /content/week3_project/data/processed/prepared_text.jsonl
File size: 16638 bytes
Validation: PASS
First record: {'record_id': 'fai_fyp_guidelines_synthetic-0001', 'text': 'Faculty of Artificial Intelligence - Final Year Project\n (FYP) Guidelines (Synthetic)\nSYNTHETIC PRACTICE DATA - created for the AI Engineering Topic 3 lab. Not official UTM\ncontent.\nVersion 1.0 (synthetic), Semester 1 2026/2027. Prepared for practice use by the FYPilot project.', 'source': 'fai_fyp_guidelines_synthetic.pdf', 'source_type': 'guidelines', 'source_sha256': '5b3722e5a236a7a6088271db6ebe4d4a4071a646fa942558a17b4643d661db0f', 'processing': 'paragraph-aware-char-800; restore_heading_breaks=True; whitespace-normalised', 'page': None}


In [9]:
# B3 BASIC OUTPUT CONTRACT — REQUIRED FOR ALL PATHWAYS
assert prepared_path.exists(), 'Prepared output missing.'
assert prepared_path.stat().st_size > 0, 'Prepared output is empty.'
print('Output:', prepared_path, 'Size:', prepared_path.stat().st_size)
print('Validation: file exists and is non-empty; verify content manually too.')

# Manual content check: one record from each source type
for key in SOURCES:
    r = next(x for x in exported_records if x["source_type"] == key)
    print(f"\n[{key}] {r['record_id']} ({len(r['text'])} chars):\n{r['text'][:300]}")

Output: /content/week3_project/data/processed/prepared_text.jsonl Size: 16638
Validation: file exists and is non-empty; verify content manually too.

[guidelines] fai_fyp_guidelines_synthetic-0001 (278 chars):
Faculty of Artificial Intelligence - Final Year Project
 (FYP) Guidelines (Synthetic)
SYNTHETIC PRACTICE DATA - created for the AI Engineering Topic 3 lab. Not official UTM
content.
Version 1.0 (synthetic), Semester 1 2026/2027. Prepared for practice use by the FYPilot project.

[past_fyps] past_fyp_abstracts_synthetic-0001 (191 chars):
SYNTHETIC PRACTICE DATA - created for the AI Engineering Topic 3 lab. Not official UTM content.
Source: synthetic records modelled on UTM ePrints abstract pages (title, year, area, abstract).

[staff_profiles] staff_research_profiles_synthetic-0001 (597 chars):
SYNTHETIC PRACTICE DATA - created for the AI Engineering Topic 3 lab. Not official UTM content.
Source: synthetic records modelled on UTM staff profile and FAI research group pages. Lecture

### Step B4 — Lineage and manifest
**B4.1 — Register Pathway A** (one entry per source document)

In [10]:
# Register Pathway A — PDF/TXT
for key, f in selected.items():
    register_lineage(source=f, output=prepared_path, processing=approach)

Registered: fai_fyp_guidelines_synthetic.pdf -> prepared_text.jsonl
Registered: past_fyp_abstracts_synthetic.txt -> prepared_text.jsonl
Registered: staff_research_profiles_synthetic.txt -> prepared_text.jsonl


In [11]:
# CELL 6A — GENERATE COMMON LINEAGE MANIFEST
# REQUIRED: Run after all selected pathways are registered
assert lineage_entries, "No lineage entries registered. Register at least one selected pathway first."
lineage_records = []

for i, entry in enumerate(lineage_entries.values(), 1):
    source = entry["source"]
    output = entry["output"]
    processing = entry["processing"]

    # Validate registered files again
    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert output.stat().st_size > 0, "Empty output"

    lineage_records.append({
        "source_id": f"S{i:02d}",
        "source_filename": str(source.relative_to(RAW_DIR.resolve())),
        "source_sha256": source_hash(source),
        "prepared_output": str(output.relative_to(BASE_DIR.resolve())),
        "prepared_sha256": source_hash(output),
        "processing_note": processing,
        "source_version": "Synthetic practice data v1.0 (2026-11-02)",
        "access_rights": "Synthetic, created by the project team; shareable. Real sources: public UTM pages, rights to be verified",
        "status": "Prepared; quality review pending"
    })

# Reuse the common manifest writer from Cell 4
manifest = save_manifest(lineage_records)

print("\nCOMMON LINEAGE MANIFEST GENERATED")
print("Total lineage entries:", len(manifest["records"]))
print("Saved to:", BASE_DIR / "data" / "week3_manifest.json")

Saved lineage: /content/week3_project/data/week3_manifest.json

COMMON LINEAGE MANIFEST GENERATED
Total lineage entries: 3
Saved to: /content/week3_project/data/week3_manifest.json


In [12]:
# CELL 6B — VERIFY COMMON LINEAGE MANIFEST
# REQUIRED: Run after Cell 6A
import json
manifest_path = BASE_DIR / "data" / "week3_manifest.json"
assert manifest_path.is_file(), "Common lineage manifest not found."
with manifest_path.open("r", encoding="utf-8") as file:
    saved_manifest = json.load(file)
saved_records = saved_manifest.get("records", [])
assert saved_records, "No lineage records found in saved manifest."
assert len(saved_records) == len(lineage_entries), "Saved lineage count does not match registration."
for record in saved_records:
    source = RAW_DIR / record["source_filename"]
    output = BASE_DIR / record["prepared_output"]
    assert source.is_file(), f"Missing source: {source}"
    assert output.is_file(), f"Missing output: {output}"
    assert source_hash(source) == record["source_sha256"], f"Source checksum mismatch: {source}"
    assert source_hash(output) == record["prepared_sha256"], f"Prepared output checksum mismatch: {output}"
print("COMMON LINEAGE VALIDATION: PASS")
print("Verified records:", len(saved_records))
print("Manifest:", manifest_path)

COMMON LINEAGE VALIDATION: PASS
Verified records: 3
Manifest: /content/week3_project/data/week3_manifest.json


## Part C — Build the V1 Data Pipeline
### Step C1 — Cell 7 (REQUIRED) and Cell 8A (PDF/TXT)

In [13]:
# Cell 7 — REQUIRED: Common V1 Pipeline Utilities
# Run ONCE after completing Part B

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

# Step 1: Define standard project paths
BASE_DIR = Path("/content/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PIPE_DIR = BASE_DIR / "data" / "pipeline_v1"

PIPE_DIR.mkdir(parents=True, exist_ok=True)

# Store registered pipeline runs for repeatability verification
PIPELINE_RUNS = {}

# Step 2: Calculate SHA-256 checksum
def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

# Step 3: Resolve an explicitly selected source
def resolve_source(relative_path, folder=False):
    source = RAW_DIR / relative_path
    if folder:
        if not source.is_dir():
            raise FileNotFoundError(f"Source folder not found: {source}")
    else:
        if not source.is_file():
            raise FileNotFoundError(f"Source file not found: {source}")
    return source

# Step 4: Execute the selected V1 data pipeline
def execute_pipeline(kind, source, inspect_fn, prepare_fn, configuration):
    source = Path(source)

    # INGEST: Identify relevant source files
    if kind == "images":
        inputs = sorted(
            p for p in source.iterdir()
            if p.is_file() and p.suffix.lower() in {".png", ".jpg", ".jpeg", ".webp"}
        )
    else:
        inputs = [source]

    if not inputs or any(not p.is_file() for p in inputs):
        raise ValueError("No valid source files to ingest.")

    # INSPECT: Apply data-type-specific inspection
    inspection = inspect_fn(inputs, configuration)

    # Define output directory
    output_dir = PIPE_DIR / kind
    output_dir.mkdir(parents=True, exist_ok=True)

    # PREPARE / STORE: Apply selected transformation
    outputs, mappings, method = prepare_fn(inputs, output_dir, configuration)
    outputs = [Path(p) for p in outputs]

    # VALIDATE: Check prepared outputs
    if not outputs:
        raise ValueError("No prepared outputs were produced.")
    for output in outputs:
        if not output.is_file():
            raise FileNotFoundError(f"Prepared output missing: {output}")
        if output.stat().st_size == 0:
            raise ValueError(f"Prepared output is empty: {output}")

    # Record basic validation evidence
    validation = {
        "output_count": len(outputs),
        "output_sha256": [sha256_file(p) for p in outputs],
        "output_bytes": [p.stat().st_size for p in outputs]
    }

    # RECORD LINEAGE: Create pipeline execution record
    record = {
        "pipeline_version": "week3-v1",
        "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "data_type": kind,
        "configuration": configuration,
        "processing_method": method,
        "sources": [{"path": str(p), "sha256": sha256_file(p)} for p in inputs],
        "outputs": [str(p) for p in outputs],
        "source_output_mapping": mappings,
        "inspection": inspection,
        "validation": validation,
        "source_version": "Not verified",
        "access_rights": "Not verified — check before sharing"
    }

    # Save pipeline manifest
    manifest_path = output_dir / "pipeline_manifest.json"
    manifest_path.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")

    print(f"SUCCESS: {kind} | Inputs: {len(inputs)} | Outputs: {len(outputs)}")
    print("Manifest:", manifest_path)
    return record

# Step 5: Register and run a selected pathway
def register_and_run(kind, source, inspect_fn, prepare_fn, configuration):
    # Keep the runner for repeatability verification
    def runner():
        return execute_pipeline(kind, source, inspect_fn, prepare_fn, configuration)
    first = runner()
    PIPELINE_RUNS[kind] = {"first": first, "runner": runner}
    return first

print("Cell 7 ready.")
print("Run ONLY the selected Cell 8 pathway(s).")
print("Common pipeline utilities are available.")

Cell 7 ready.
Run ONLY the selected Cell 8 pathway(s).
Common pipeline utilities are available.


In [14]:
# Cell 8A — CONDITIONAL REQUIRED: PDF/TXT Projects
# Requires Cell 7. Uses the same configuration chosen in Part B (configuration C).
!pip -q install pypdf

import re
import json
from pypdf import PdfReader

# Step 1: Select the sources and preparation configuration (same as Part B)
SOURCES_A = {
    "documents_guidelines":     resolve_source("fai_fyp_guidelines_synthetic.pdf"),
    "documents_past_fyps":      resolve_source("past_fyp_abstracts_synthetic.txt"),
    "documents_staff_profiles": resolve_source("staff_research_profiles_synthetic.txt"),
}
CONFIG_A = {
    "chunk_size": 800,
    "method": "paragraph-aware",
    "restore_heading_breaks": True
}

def _extract(src):
    if src.suffix.lower() == ".pdf":
        pages = [page.extract_text() or "" for page in PdfReader(str(src)).pages]
        return "\n".join(pages), len(pages)
    elif src.suffix.lower() == ".txt":
        return src.read_text(encoding="utf-8"), None
    raise ValueError("Expected a PDF or TXT source")

# Step 2: Inspect document content
def inspect_documents(inputs, cfg):
    src = inputs[0]
    text, page_count = _extract(src)
    if not text.strip():
        raise ValueError("No extractable text found. Scanned PDFs may require OCR.")
    return {"source": src.name, "characters": len(text), "pages": page_count}

# Step 3: Define document preparation
def prepare_documents(inputs, out, cfg):
    src = inputs[0]
    text, _ = _extract(src)
    # Apply the same cleaning used in Part B
    text = re.sub(r"[ \t]+", " ", text).strip()
    if cfg.get("restore_heading_breaks"):
        text = re.sub(r"\n(?=\d+\. [A-Z])", "\n\n", text)

    size = int(cfg["chunk_size"])
    method = cfg["method"]
    if size <= 0:
        raise ValueError("Chunk size must be positive")

    if method == "fixed":
        chunks = [text[i:i + size] for i in range(0, len(text), size)]
    elif method == "paragraph-aware":
        chunks = []
        current = ""
        for section in re.split(r"\n\s*\n", text):
            section = section.strip()
            if not section:
                continue
            if len(section) > size:
                if current:
                    chunks.append(current)
                    current = ""
                chunks.extend(section[i:i + size] for i in range(0, len(section), size))
            elif len(current) + len(section) + (2 if current else 0) <= size:
                current += ("\n\n" if current else "") + section
            else:
                if current:
                    chunks.append(current)
                current = section
        if current:
            chunks.append(current)
    else:
        raise ValueError("Choose 'fixed' or 'paragraph-aware'")

    if not chunks:
        raise ValueError("No text chunks produced")

    # Step 4: Store prepared document records
    target = out / "prepared_text.jsonl"
    with target.open("w", encoding="utf-8") as handle:
        for i, chunk in enumerate(chunks, 1):
            record = {"id": i, "source": str(src), "text": chunk, "page": None}
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")

    # Step 5: Return source-to-output mapping
    mapping = [{"source": str(src), "output": str(target)}]
    processing_method = (
        f"{method} text chunks of {size} characters; "
        f"restore_heading_breaks={cfg.get('restore_heading_breaks', False)}"
    )
    return [target], mapping, processing_method

# Step 6: Execute and register the pipeline for each selected document source
run_A = {}
for kind, source in SOURCES_A.items():
    run_A[kind] = register_and_run(kind, source, inspect_documents, prepare_documents, CONFIG_A)
print("Cell 8A completed successfully.")

# Inspect a representative prepared record (a non-empty file alone is not proof of correctness)
sample = (PIPE_DIR / "documents_past_fyps" / "prepared_text.jsonl").read_text(encoding="utf-8").splitlines()[1]
print("\nSample prepared record:", sample[:400])

SUCCESS: documents_guidelines | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents_guidelines/pipeline_manifest.json
SUCCESS: documents_past_fyps | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents_past_fyps/pipeline_manifest.json
SUCCESS: documents_staff_profiles | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents_staff_profiles/pipeline_manifest.json
Cell 8A completed successfully.

Sample prepared record: {"id": 2, "source": "/content/week3_project/data/raw/past_fyp_abstracts_synthetic.txt", "text": "Record ID: PFYP-2023-01\nTitle: Deep Learning-Based Durian Leaf Disease Detection for Smallholder Farmers\nYear: 2023\nResearch area: Computer vision\nAbstract: Durian leaf diseases reduce yield for smallholder farmers who lack access to plant pathologists. This project developed a convolutional neural


In [15]:
# Cell 9 — REQUIRED: Repeatability and Provenance Verification
# Run after your selected Cell 8A–8F pathway(s).

# Step 1: Ensure at least one pipeline has been registered
assert PIPELINE_RUNS, "No pipeline runs registered. Run at least one selected Cell 8 pathway first."
print("--- Repeatability Verification ---")

# Step 2: Rerun each selected pipeline
for kind, entry in PIPELINE_RUNS.items():
    first = entry["first"]
    # Execute the same pipeline again
    second = entry["runner"]()
    # Step 3: Verify input provenance
    assert first["sources"] == second["sources"], f"Source fingerprint changed for {kind}"
    # Step 4: Verify prepared output fingerprints
    assert first["validation"] == second["validation"], f"Prepared output changed for {kind}"
    # Step 5: Verify source-to-output mapping
    assert first["source_output_mapping"] == second["source_output_mapping"], (
        f"Source-to-output mapping changed for {kind}"
    )
    print(f"PASS: {kind} — source fingerprints, prepared outputs and mappings match")

# Step 6: Display final verification message
print("\nAll selected pipelines passed repeatability smoke checks.")

--- Repeatability Verification ---


SUCCESS: documents_guidelines | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents_guidelines/pipeline_manifest.json
PASS: documents_guidelines — source fingerprints, prepared outputs and mappings match
SUCCESS: documents_past_fyps | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents_past_fyps/pipeline_manifest.json
PASS: documents_past_fyps — source fingerprints, prepared outputs and mappings match
SUCCESS: documents_staff_profiles | Inputs: 1 | Outputs: 1
Manifest: /content/week3_project/data/pipeline_v1/documents_staff_profiles/pipeline_manifest.json
PASS: documents_staff_profiles — source fingerprints, prepared outputs and mappings match

All selected pipelines passed repeatability smoke checks.


### Step C4a — Package outputs for download
Files under `/content` are temporary. This zips `data/` so it can be copied into the GitHub repo.

In [16]:
# C4a — Package Week 3 outputs (Colab download when available)
import shutil
archive = shutil.make_archive("/content/week3_outputs", "zip", BASE_DIR / "data")
print("Packaged:", archive)
for p in sorted((BASE_DIR / "data").rglob("*")):
    if p.is_file():
        print(" ", p.relative_to(BASE_DIR), p.stat().st_size, "bytes")
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Not running in Colab — copy the files above into the repo manually.")

Packaged: /content/week3_outputs.zip
  data/pipeline_v1/documents_guidelines/pipeline_manifest.json 1280 bytes
  data/pipeline_v1/documents_guidelines/prepared_text.jsonl 4145 bytes
  data/pipeline_v1/documents_past_fyps/pipeline_manifest.json 1280 bytes
  data/pipeline_v1/documents_past_fyps/prepared_text.jsonl 6101 bytes
  data/pipeline_v1/documents_staff_profiles/pipeline_manifest.json 1310 bytes
  data/pipeline_v1/documents_staff_profiles/prepared_text.jsonl 2251 bytes
  data/processed/prepared_text.jsonl 16638 bytes
  data/raw/fai_fyp_guidelines_synthetic.pdf 4568 bytes
  data/raw/past_fyp_abstracts_synthetic.txt 5059 bytes
  data/raw/staff_research_profiles_synthetic.txt 1872 bytes
  data/week3_manifest.json 2227 bytes
Not running in Colab — copy the files above into the repo manually.
